<a href="https://colab.research.google.com/github/santimndez/tfm/blob/main/src/notebooks/sam3_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Configuration
# --------- CONFIG ---------
VIDEO_FOLDER = './videos'
VIDEO_IN  = "videos/b1.mp4"              # vídeo de entrada
VIDEO_OUT = "videos/s1b_sam3.mp4"
MASKS_OUT = "masks"                  # carpeta para máscaras (se crea automáticamente)
TEXT_PROMPT = "ping pong ball"       # prompt para segmentación
MAX_FRAMES = None                    # por ejemplo 200 para pruebas; None = todo el vídeo
TARGET_SHORT_SIDE = 480              # reescalar lado corto para ahorrar memoria (baja si hace falta)
WINDOW_SIZE = 120                    # ventana temporal de frames para ahorrar memoria
# ---------------------------


In [ ]:
#@title Download video
import gdown

# FILE_ID_VIDEO = '1LTl8f8-kBhHE0T7qvj84LotfeAiufxVL' # S1A
FILE_ID_REQUIREMENTS = '16DUdAKrJe25gHmaGjJBDwVZkXy2b6vt2'

gdown.download(id=FILE_ID_REQUIREMENTS, output='./requirements.txt', quiet=False)
!pip install -r requirements.txt


Downloading...
From: https://drive.google.com/uc?id=16DUdAKrJe25gHmaGjJBDwVZkXy2b6vt2
To: /content/requirements.txt
100%|██████████| 1.14k/1.14k [00:00<00:00, 1.05MB/s]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.2/516.2 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

NameError: name 'os' is not defined

In [ ]:
import gdown
import os

FILE_ID_VIDEO = '15Y4XSwyCMjs3Q4xngo0y1kDmAirZZw20' # S1B

os.makedirs(VIDEO_FOLDER, exist_ok=True)
gdown.download(id=FILE_ID_VIDEO, output=VIDEO_IN, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=15Y4XSwyCMjs3Q4xngo0y1kDmAirZZw20
To: /content/videos/b1.mp4
100%|██████████| 105M/105M [00:01<00:00, 79.8MB/s] 


'videos/b1.mp4'

In [ ]:

import cv2
import torch
import numpy as np
from accelerate import Accelerator
from transformers import Sam3VideoModel, Sam3VideoProcessor
import os
import zipfile
import time

def zip_folder(folder_path, output_path=""):
    """
    Comprime recursivamente el contenido de `folder_path` en un archivo .zip.
    """
    if output_path=="":
        output_path = folder_path + ".zip"
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Relativizar la ruta para no guardar las rutas absolutas
                arcname = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, arcname)

In [ ]:
# Log in to Hugging Face with token secret
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [ ]:

def main():
    accelerator = Accelerator()
    device = accelerator.device
    print("Dispositivo:", device)

    # --------------------------------------------------------------
    # 1. Modelo SAM3 EN FP32 (NO tocar dtype -> así no produce fallos)
    # --------------------------------------------------------------
    model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device)
    processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

    # --------------------------------------------------------------
    # 2. Cargar vídeo con OpenCV (evitamos PyAV)
    # --------------------------------------------------------------
    cap = cv2.VideoCapture(VIDEO_IN)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir {VIDEO_IN}")

    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    fps = orig_fps if orig_fps and orig_fps > 0 else 25

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Escalado opcional para ahorrar VRAM
    if orig_h < orig_w:
        scale = TARGET_SHORT_SIDE / float(orig_h)
    else:
        scale = TARGET_SHORT_SIDE / float(orig_w)
    scale = min(scale, 1.0)

    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (new_w, new_h))
    os.makedirs(MASKS_OUT, exist_ok=True)    # Crear carpeta para guardar las máscaras

    # --------------------------------------------------------------
    # 3. Inicializar sesión SAM3 EN FP32 (si pones bf16/half casca)
    # --------------------------------------------------------------
    def sam3sesion(processor, inference_device, processing_device, video_storage_device, prompt=TEXT_PROMPT):
      streaming_inference_session = processor.init_video_session(
          inference_device=device,
          processing_device=processing_device,        # mantener CPU reduce VRAM
          video_storage_device=video_storage_device,  # "cpu",
          dtype=torch.float32,          # *** CLAVE: FP32 ***
      )

      # Añadir prompt de texto
      streaming_inference_session = processor.add_text_prompt(
          inference_session=streaming_inference_session,
          text=prompt,
      )
      return streaming_inference_session

    streaming_inference_session = sam3sesion(processor, device, 'cpu', 'cpu')
    # --------------------------------------------------------------
    # 4. Procesamiento FRAME A FRAME
    # --------------------------------------------------------------
    frame_idx = 0
    tic = time.time()
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
            break

        # Reescalar si procede
        if scale != 1.0:
            frame_bgr = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        # ---- preparar entrada para SAM3 (siempre FP32) ----
        inputs = processor(images=frame_rgb, return_tensors="pt")
        frame_tensor = inputs["pixel_values"][0].to(device=device, dtype=torch.float32)

        # ---- inferencia SAM3 en streaming ----
        with torch.no_grad():
          model_outputs = model(
              inference_session=streaming_inference_session,
              frame=frame_tensor,
              reverse=False,
          )

        # ---- post-procesado SAM3 ----
        processed_outputs = processor.postprocess_outputs(
            streaming_inference_session,
            model_outputs,
            original_sizes=inputs.original_sizes,
        )

        masks = processed_outputs["masks"]  # (N_obj, H, W)

        if masks.numel() == 0:
            frame_vis = frame_bgr
            combined_mask = None
        else:
            combined_mask = (masks > 0).any(dim=0).cpu().numpy().astype(np.uint8)

            overlay = frame_bgr.copy()
            overlay[combined_mask == 1] = (0, 255, 0)

            frame_vis = cv2.addWeighted(overlay, 0.5, frame_bgr, 0.5, 0)

        # Guardar máscara:
        if combined_mask is not None:
          cv2.imwrite(os.path.join(MASKS_OUT, f"mask_{frame_idx:06d}.png"), combined_mask * 255)

        writer.write(frame_vis)

        frame_idx += 1
        if frame_idx % 100 == 0:
            toc = time.time()
            print(f"Procesados {frame_idx} frames...  {toc-tic} s")
            tic = time.time()
        if WINDOW_SIZE is not None and frame_idx % WINDOW_SIZE == 0:
          del streaming_inference_session
          torch.cuda.empty_cache()

          streaming_inference_session = sam3sesion(processor, device, 'cpu', 'cpu')

    cap.release()
    writer.release()
    print("\n✔ Vídeo generado:", VIDEO_OUT)
    print("✔ Máscaras guardadas en:", MASKS_OUT)
    # Comprimir las máscaras en un ZIP
    zip_folder(MASKS_OUT)
    print("✔ Máscaras comprimidas en:", MASKS_OUT + ".zip")


In [ ]:
main()


Dispositivo: cuda


Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

Fetching 0 files: 0it [00:00, ?it/s]

Failed to load cv_utils kernel (your torch/cuda setup may not be supported): Cannot install kernel from repo kernels-community/cv_utils (revision: main). NMS post-processing, hole filling, and sprinkle removal will be skipped.


Procesados 100 frames...  240.47592544555664 s
Procesados 200 frames...  258.8034944534302 s
Procesados 300 frames...  246.80832600593567 s
Procesados 400 frames...  262.3199245929718 s
Procesados 500 frames...  257.81746768951416 s
Procesados 600 frames...  263.6018750667572 s
Procesados 700 frames...  262.58392667770386 s
Procesados 800 frames...  259.7887074947357 s
Procesados 900 frames...  260.07111144065857 s
Procesados 1000 frames...  263.3933868408203 s
Procesados 1100 frames...  259.8369126319885 s
Procesados 1200 frames...  260.48077487945557 s
Procesados 1300 frames...  259.33043098449707 s
Procesados 1400 frames...  259.7796051502228 s
Procesados 1500 frames...  259.25044655799866 s
Procesados 1600 frames...  261.9152629375458 s
Procesados 1700 frames...  258.63597321510315 s
Procesados 1800 frames...  259.9313921928406 s
Procesados 1900 frames...  258.9599013328552 s
Procesados 2000 frames...  259.84917426109314 s
